# Data Reading from Bronze Layer

In [0]:
%run ../00-common/01.environment_config

In [0]:
bronze_table = f"{catalog_name}.{bronze_schema}.circuits"
sliver_table = f"{catalog_name}.{silver_schema}.circuits"

In [0]:
sliver_table

'formula1.silver.circuits'

In [0]:
circuits_df= spark.read.table(bronze_table)



In [0]:
circuits_df.show(5)

+---------+--------------------+--------------------+--------+--------+----------+---------+--------------------+--------------------+
|circuitId|                 url|         circuitName|     lat|    long|  locality|  country| ingestion_timestamp|        source_files|
+---------+--------------------+--------------------+--------+--------+----------+---------+--------------------+--------------------+
|     NULL|https://en.wikipe...|circuit gilles vi...|    45.5|-73.5228|  montreal|   Canada|2026-09-29 04:39:...|dbfs:/Volumes/for...|
|     NULL|https://en.wikipe...|losail internatio...|   25.49| 51.4542|    lusail|    Qatar|2026-09-29 04:39:...|dbfs:/Volumes/for...|
| adelaide|https://en.wikipe...|adelaide street c...|-34.9272| 138.617|  adelaide|Australia|2026-09-29 04:39:...|dbfs:/Volumes/for...|
| ain-diab|https://en.wikipe...|            ain diab| 33.5786| -7.6875|casablanca|  Morocco|2026-09-29 04:39:...|dbfs:/Volumes/for...|
|  aintree|https://en.wikipe...|             aintree| 5

# Data Cleaning

In [0]:
from pyspark.sql.functions import *

In [0]:
circuits_selected_df=circuits_df.select(
                                        col("circuitId"),
                                        col("circuitName"),
                                        col("lat"),
                                        col("long"),
                                        col("locality"),
                                        col("country"),
                                        col("ingestion_timestamp"),
                                        col("source_files")
                                    )

In [0]:
#circuits_renamed_df=(circuits_selected_df.withColumnRenamed("circuitId","circuit_id") 
                  # .withColumnRenamed("circuitName","circuit_name") 
                   #.withColumnRenamed("lat","latitude") 
                   #.withColumnRenamed("long","longitude") 
                   #.withColumnRenamed("locality","city") 
                   #)
#circuits_renamed_df.show(5)

In [0]:
circuits_renamed_df = circuits_selected_df.withColumnsRenamed(
   {
        "circuitId": "circuit_id",
       "circuitName": "circuit_name",
       "lat": "latitude",
       "long": "longitude",
       "locality": "city"
   }
)

In [0]:
circuits_renamed_df.show(5)

+----------+--------------------+--------+---------+----------+---------+--------------------+--------------------+
|circuit_id|        circuit_name|latitude|longitude|      city|  country| ingestion_timestamp|        source_files|
+----------+--------------------+--------+---------+----------+---------+--------------------+--------------------+
|      NULL|circuit gilles vi...|    45.5| -73.5228|  montreal|   Canada|2026-09-29 04:39:...|dbfs:/Volumes/for...|
|      NULL|losail internatio...|   25.49|  51.4542|    lusail|    Qatar|2026-09-29 04:39:...|dbfs:/Volumes/for...|
|  adelaide|adelaide street c...|-34.9272|  138.617|  adelaide|Australia|2026-09-29 04:39:...|dbfs:/Volumes/for...|
|  ain-diab|            ain diab| 33.5786|  -7.6875|casablanca|  Morocco|2026-09-29 04:39:...|dbfs:/Volumes/for...|
|   aintree|             aintree| 53.4769| -2.94056| liverpool|       UK|2026-09-29 04:39:...|dbfs:/Volumes/for...|
+----------+--------------------+--------+---------+----------+---------

# Data Validation

In [0]:
circuits_valid_df = circuits_renamed_df .filter(
    col("circuit_id").isNotNull()
)

# Removing duplicates Circuit ID

In [0]:
circuits_distincts_df = circuits_valid_df.dropDuplicates(["circuit_id"])

# Making Circuit Name and City in INITCAP

In [0]:
circuits_final_df = (
                    circuits_distincts_df
                     .withColumn("circuit_name", initcap(col("circuit_name")))
                     .withColumn("city", initcap(col("city")))
                    )

# Write Data as Delta Table in Silver Schema

In [0]:
circuits_final_df.write.mode("overwrite").format("delta").saveAsTable(sliver_table)

In [0]:
display(spark.table(sliver_table))

circuit_id,circuit_name,latitude,longitude,city,country,ingestion_timestamp,source_files
adelaide,Adelaide Street Circuit,-34.9272,138.617,Adelaide,Australia,2026-09-29T04:39:41.381468Z,dbfs:/Volumes/formula1/landing/files/circuits.csv
ain-diab,Ain Diab,33.5786,-7.6875,Casablanca,Morocco,2026-09-29T04:39:41.381468Z,dbfs:/Volumes/formula1/landing/files/circuits.csv
aintree,Aintree,53.4769,-2.94056,Liverpool,UK,2026-09-29T04:39:41.381468Z,dbfs:/Volumes/formula1/landing/files/circuits.csv
albert_park,Albert Park Grand Prix Circuit,-37.8497,144.968,Melbourne,Australia,2026-09-29T04:39:41.381468Z,dbfs:/Volumes/formula1/landing/files/circuits.csv
americas,Circuit Of The Americas,30.1328,-97.6411,Austin,USA,2026-09-29T04:39:41.381468Z,dbfs:/Volumes/formula1/landing/files/circuits.csv
anderstorp,Scandinavian Raceway,57.2653,13.6042,Anderstorp,Sweden,2026-09-29T04:39:41.381468Z,dbfs:/Volumes/formula1/landing/files/circuits.csv
avus,Avus,52.4806,13.2514,Berlin,Germany,2026-09-29T04:39:41.381468Z,dbfs:/Volumes/formula1/landing/files/circuits.csv
bahrain,Bahrain International Circuit,26.0325,50.5106,Sakhir,Bahrain,2026-09-29T04:39:41.381468Z,dbfs:/Volumes/formula1/landing/files/circuits.csv
baku,Baku City Circuit,40.3725,49.8533,Baku,Azerbaijan,2026-09-29T04:39:41.381468Z,dbfs:/Volumes/formula1/landing/files/circuits.csv
boavista,Circuito Da Boavista,41.1705,-8.67325,Oporto,Portugal,2026-09-29T04:39:41.381468Z,dbfs:/Volumes/formula1/landing/files/circuits.csv
